In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto MLDM')

    GIT_BRANCH = 'incremento-punti-dataset'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

# 03 - Creazione e salvataggio dataset
- estrazione dei valori spettrali Sentinel-2 per tutti i punti campionati (campionamento diretto dai GeoTIFF dell'area)
- feature engineering (serie temporali NDVI/NDWI e indici fenologici per la classificazione)

#### Caricamento ground truth e legenda

In [ ]:
import json
import pandas as pd

# definizione dei percorsi di input
ground_truth_path = DATA_DIR / "interim" / "points.json"
sentinel2_data_path = DATA_DIR / "processed" / "sentinel2_capitanata_area"
legend_path = DATA_DIR / "processed" / "legend.json"

# caricamento del file con le coordinate e i codici delle colture
ground_truth_df = pd.read_json(ground_truth_path)
print(f"Campi della Capitanata caricati: {len(ground_truth_df)}")

# caricamento della legenda con i nomi delle colture
legend = {}
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)
    print(f"Legenda caricata con successo ({len(legend)} voci).")
else:
    print(f"⚠️ Legenda non trovata in {legend_path}.")

#### Estrazione dei valori spettrali dai GeoTIFF dell'area Capitanata
Campionamento ad alta velocità tramite finestra 3x3 (mediana spaziale) direttamente dai raster mensili.

In [ ]:
import re
import numpy as np
import rasterio
from rasterio.warp import transform
from tqdm import tqdm

# elenco dei file raster mensili disponibili
tif_files = sorted(list(sentinel2_data_path.glob("*.tif")))

# mappatura indici delle 6 bande (1-based index per rasterio)
# ordine: 1: B02 (Blu), 2: B03 (Verde), 3: B04 (Rosso), 4: B08 (NIR), 5: B11 (SWIR1), 6: B12 (SWIR2)
BAND_INDEX_MAP = {
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B08": 4,
    "B11": 5,
    "B12": 6,
}

# struttura dati per memorizzare i valori per ogni punto e per ciascun mese (1-12)
point_monthly_values = {idx: {m: {} for m in range(1, 13)} for idx in range(len(ground_truth_df))}

lons = ground_truth_df["lon"].values
lats = ground_truth_df["lat"].values
years = ground_truth_df["year"].astype(int).values

for tif_path in tqdm(tif_files, desc="Elaborazione raster mensili"):
    # Estrae l'anno e il mese dal nome del file (es. capitanata_2022_05.tif)
    match = re.search(r"(\d{4})[-_](\d{2})", tif_path.name)
    if not match:
        continue
    tif_year = int(match.group(1))
    mo = int(match.group(2))

    # Filtra ed elabora ESCLUSIVAMENTE i punti ground truth di quello specifico anno
    point_indices = np.where(years == tif_year)[0]
    if len(point_indices) == 0:
        continue

    with rasterio.open(tif_path) as src:
        # converte le coordinate gps dei soli punti di quest'anno nel CRS del raster
        xs, ys = transform("EPSG:4326", src.crs, lons[point_indices], lats[point_indices])
        n_bands = src.count

        for idx, x, y in zip(point_indices, xs, ys):
            r_c, c_c = src.index(x, y)

            # verifica che il punto ricada dentro i confini dell'immagine
            if 0 <= r_c < src.height and 0 <= c_c < src.width:
                # campionamento su singolo pixel 10x10 m
                window = ((r_c, r_c + 1), (c_c, c_c + 1))
                pixel_data = src.read(window=window)

                for b_name, b_idx in BAND_INDEX_MAP.items(): 
                    if b_idx <= n_bands:                     
                        val = pixel_data[b_idx - 1, 0, 0]    
                        if val > 0:                                                         
                            point_monthly_values[idx][mo][b_name] = float(val)

print("Estrazione dei dati satellitari completata per tutti i punti!")

#### Feature engineering e Creazione dataset

In [ ]:
from src.feature_engineering import extract_features_from_monthly_stack, FEATURE_LIST

# ciascun punto ha un ciclo annuale di 12 mesi
min_required_months = 6
print(f"Soglia minima mesi validi richiesta per punto: {min_required_months} (su 12 mesi annuali)")

# preparazione dei valori estratti in un array array 3D (12 mesi, 6 bande, n_campi)
band_list = ["B02", "B03", "B04", "B08", "B11", "B12"] 
n_points = len(ground_truth_df)
monthly_points = np.zeros((12, 6, n_points), dtype=np.float32)

for idx in range(n_points):
    for m in range(1, 13):
        for b_idx, b_name in enumerate(band_list):
            monthly_points[m-1, b_idx, idx] = point_monthly_values[idx][m].get(b_name, 0.0)

# scarto di tutti i campi con meno di 6 mesi osservati
valid_months_per_point = np.sum(monthly_points[:, 2, :] > 0, axis=0)                                                        
valid_mask = valid_months_per_point >= min_required_months   
discarded_few_months = int(np.sum(~valid_mask))              
                                                                 
monthly_points_valid = monthly_points[:, :, valid_mask]      
filtered_ground_truth = ground_truth_df[valid_mask].reset_index(drop=True) 

# calcolo di tutte le features                                      
_, features_matrix = extract_features_from_monthly_stack(monthly_points_valid)            

# creazione del DataFrame finale                          
features_df = pd.DataFrame(features_matrix, columns=FEATURE_LIST) 
                                                                 
metadata_df = pd.DataFrame({                                 
    "ID_Campo": filtered_ground_truth.index,                       
    "Year": filtered_ground_truth["year"].astype(int),           
    "Ground_Truth": filtered_ground_truth["code"].astype(int),     
    "Crop_Name": filtered_ground_truth["code"].astype(str).map(legend)                                                    
})                                                           
                                                                
final_df = pd.concat([metadata_df, features_df], axis=1)     
                                                                
print(f"Dataset generato con successo: {final_df.shape[0]}")
print(f"  • Punti Capitanata considerati: {len(ground_truth_df)}") 
print(f"  • Punti scartati per osservazioni insufficienti: {discarded_few_months}") 
print(f"  ✅ Punti validi finali nel dataset: {len(final_df)}")

#### Salvataggio e report del dataset finale

In [ ]:
dataset_dir = DATA_DIR / 'processed' / 'dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

parquet_path = dataset_dir / 'dataset.parquet'
csv_path = dataset_dir / 'dataset.csv'

final_df.to_parquet(parquet_path, index=False)
final_df.to_csv(csv_path, index=False)

print(f"File salvati con successo in:")
print(f"   • Parquet: {parquet_path.resolve()}")
print(f"   • CSV:     {csv_path.resolve()}")
print(f"\nDimensioni tabella finale: {final_df.shape[0]} righe x {final_df.shape[1]} colonne")

# controllo campioni estratti per ciascuna coltura
print(f"Distribuzione dei campioni estratti per classe:")
display(final_df['Crop_Name'].value_counts())

print(f"\nDistribuzione dei campioni per anno:")
display(final_df['Year'].value_counts().sort_index())

# anteprima delle prime 5 righe
display(final_df.head())